In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import random


In [ ]:

# 数据预处理：将图像转换为张量并归一化到 [0,1]
transform = transforms.Compose([
    transforms.Resize((224, 224)),          # 将图像短边或长边缩放到224×224（可能变形）
    transforms.ToTensor(),                   # 转换为Tensor并缩放到[0,1]
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet标准化参数
                         std=[0.229, 0.224, 0.225])
])

class MyDataset(Dataset):
    def __init__(self, samples, transform=None, preload_image=True):
        self.samples = samples  # [(imgpath, identity, age)]
        self.transform = transform
        self.preload_image = preload_image

        if preload_image:
            total_count = len(samples)
            total_byten = total_count // 10
            print(f'image loading... total {total_count}')
            for idx, sample in enumerate(self.samples):
                image_path = sample[0]
                image = self.load_image(image_path)
                self.samples[idx] = (image,sample[1],sample[2])
                
                if idx % total_byten == 0:
                    print(str(idx)+' loaded!')
            print('image loading complete!')

        self.id_to_label = {}
        label = 0
        for sample in samples:
            id = sample[1]
            if not id in self.id_to_label:
                self.id_to_label[id] = label
                label += 1

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        if self.preload_image:
            image = sample[0]
        else:
            image_path = sample[0]
            image = self.load_image(image_path)

        identity = sample[1]
        label = self.id_to_label[identity]

        age = sample[2]
        if age == -1:
            age_label = -1
        else:
            age_label = age // 5
        return image, label, age_label
    
    def load_image(self, path):
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image


In [ ]:

def get_dataset(path):

    # 读取CSV文件并分割数据集
    with open(path, 'r') as f:
        lines = f.readlines()
    lines = lines[1:]  # 跳过表头

    train_data = []
    test_data = []

    temp_data = []
    temp_shootdates = []

    cur_id = None
    for line in lines:
        parts = line.strip().split(',')
        path = parts[0]  # 图像路径
        id = parts[1]
        dob = parts[3]
        shootdate = parts[8]

        image_path = './data/Images/'+id+'/'+path

        if len(shootdate) >= 4 and len(dob) >= 4:
            age = int(shootdate[:4]) - int(dob[:4])
        else:
            age = -1 # 无年龄信息

        # 当遇到新的id时，处理之前的id的数据
        if cur_id is not None and id != cur_id:
            temp_count = len(temp_data)
            train_count = int(temp_count * 0.8) # 训练集数量

            cur_tcount = 0
            cur_tshootdate = None

            # 训练集取偏幼年或偏老年
            rand = random.randint(0,1)
            # 幼年
            if rand == 0:
                for tdata, tshootdate in zip(temp_data, temp_shootdates):
                    if cur_tcount >= train_count and tshootdate != cur_tshootdate: # 确保同一拍摄日期的图像不会被同时分到训练集和测试集中
                        break

                    train_data.append(tdata)
                    cur_tcount += 1
                    cur_tshootdate = tshootdate
                
                test_data.extend(temp_data[cur_tcount:])

            # 老年
            else:
                reversed_temp_data = list(reversed(temp_data))
                reversed_temp_shootdates = list(reversed(temp_shootdates))

                for tdata, tshootdate in zip(reversed_temp_data,  reversed_temp_shootdates):
                    if cur_tcount >= train_count and tshootdate != cur_tshootdate: # 确保同一拍摄日期的图像不会被同时分到训练集和测试集中
                        break
                    
                    train_data.append(tdata)
                    cur_tcount += 1
                    cur_tshootdate = tshootdate
                
                test_data.extend(reversed_temp_data[cur_tcount:])

            temp_data = []
            temp_shootdates = []

        cur_id = id

        temp_data.append((image_path, id, age))
        temp_shootdates.append(shootdate)

    train_dataset = MyDataset(train_data, transform=transform)
    test_dataset = MyDataset(test_data, transform=transform) 
    return train_dataset, test_dataset

train_dataset, test_dataset = get_dataset('./data/MFD_metadatas.csv')
# 创建数据加载器
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

image loading... total 24574
0 loaded!
2457 loaded!
4914 loaded!
7371 loaded!
9828 loaded!
12285 loaded!
14742 loaded!
17199 loaded!
19656 loaded!
22113 loaded!
24570 loaded!
image loading complete!
image loading... total 4918
0 loaded!
491 loaded!
982 loaded!
1473 loaded!
1964 loaded!
2455 loaded!
2946 loaded!
3437 loaded!
3928 loaded!
4419 loaded!
4910 loaded!
image loading complete!


In [4]:
# 特征提取器
class ResNet50FeatureExtractor(nn.Module):
    def __init__(self, embedding_size=512, pretrained=True):
        super().__init__()
        # 加载预训练的 ResNet50
        resnet = models.resnet50(pretrained=pretrained)
        # 去掉原始的全连接层和池化层（我们保留全局平均池化）
        self.features = nn.Sequential(*list(resnet.children())[:-1])  # 输出形状: (batch, 2048, 1, 1)
        # 可选：添加一个全连接层将 2048 维映射到 embedding_size
        self.fc = nn.Linear(2048, embedding_size)
        # 初始化 fc 层
        nn.init.xavier_normal_(self.fc.weight)
        nn.init.constant_(self.fc.bias, 0)

    def forward(self, x):
        # x: (batch, 3, 224, 224) 或 (batch, 3, 112, 112) 需要与预训练输入尺寸匹配
        # 如果输入是 112x112，可能需要调整第一层卷积 stride 或 resize 输入，这里假设输入已 resize 到 224
        x = self.features(x)          # (batch, 2048, 1, 1)
        x = x.view(x.size(0), -1)     # (batch, 2048)
        x = self.fc(x)                 # (batch, embedding_size)
        return x                       # 未归一化的特征向量

In [5]:
# 身份分类头
class ArcFaceHead(nn.Module):
    def __init__(self, embedding_size=512, num_classes=1000, margin=0.5, scale=64):
        super().__init__()
        self.embedding_size = embedding_size
        self.num_classes = num_classes
        self.margin = margin
        self.scale = scale
        # 权重矩阵，每一列代表一个类中心
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_size))
        nn.init.xavier_normal_(self.weight)

    def forward(self, features, labels):
        """
        features: (batch, embedding_size)  未归一化的特征
        labels: (batch,)  每个样本的真实类别索引
        返回: 用于交叉熵损失的 logits (batch, num_classes)
        """
        # 归一化特征和权重
        features = F.normalize(features, p=2, dim=1)
        weight = F.normalize(self.weight, p=2, dim=1)  # (num_classes, embedding_size)

        # 计算余弦相似度矩阵 (batch, num_classes)
        cosine = F.linear(features, weight)  # features @ weight.T

        # 获取每个样本真实类别的余弦值
        batch_size = features.size(0)
        cosine_theta = cosine[torch.arange(batch_size), labels].view(-1, 1)  # (batch, 1)

        # 计算 sin(θ) 用于添加角度 margin
        sin_theta = torch.sqrt(1.0 - torch.pow(cosine_theta, 2))

        # 将 margin 转换为与 features 相同设备和类型的张量
        cos_m = torch.cos(torch.tensor(self.margin, dtype=features.dtype, device=features.device))
        sin_m = torch.sin(torch.tensor(self.margin, dtype=features.dtype, device=features.device))

        # cos(θ + m) = cosθ cos m - sinθ sin m
        cos_theta_m = cosine_theta * cos_m - sin_theta * sin_m

        # 构造新的 logits：真实类别位置放 cos(θ+m)，其余保持 cosθ
        logits = cosine * self.scale  # 先乘以 scale
        logits.scatter_(1, labels.view(-1, 1), cos_theta_m * self.scale)

        return logits

In [6]:
# 年龄预测器
class AgePredictor(nn.Module):
    def __init__(self, embedding_size=512, num_age_groups=10):
        super().__init__()
        self.fc1 = nn.Linear(embedding_size, 128)
        self.fc2 = nn.Linear(128, num_age_groups)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)          # 输出 logits，未经过 softmax
        return x

In [7]:
# 梯度反转层
class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_, None

def grad_reverse(x, lambda_=1.0):
    return GradientReversal.apply(x, lambda_)

In [8]:
# 训练器
class CrossAgeFaceModel(nn.Module):
    def __init__(self, embedding_size=512, num_classes=1000, num_age_groups=10,
                 margin=0.5, scale=64, age_loss_weight=0.1, rev_lambda=0.1):
        super().__init__()
        self.embedding_size = embedding_size
        self.age_loss_weight = age_loss_weight
        self.rev_lambda = rev_lambda

        # 特征提取器
        self.backbone = ResNet50FeatureExtractor(embedding_size=embedding_size)

        # ArcFace 头
        self.arcface_head = ArcFaceHead(embedding_size, num_classes, margin, scale)

        # 年龄预测器（训练时使用）
        self.age_predictor = AgePredictor(embedding_size, num_age_groups)

    def forward(self, images, labels=None, age_labels=None):
        """
        训练阶段: 返回总损失和各个分量
        推理阶段: 仅返回特征向量
        """
        features = self.backbone(images)  # (batch, embedding_size)

        if labels is not None:  # 训练模式，需要计算损失
            # ArcFace 损失
            logits = self.arcface_head(features, labels)
            loss_arcface = F.cross_entropy(logits, labels)

            # 年龄对抗损失
            # 对特征应用梯度反转
            features_adv = grad_reverse(features, self.rev_lambda)
            age_logits = self.age_predictor(features_adv)

            # 处理缺失年龄标签
            valid_mask = age_labels != -1
            if valid_mask.any():
                valid_age_logits = age_logits[valid_mask]
                valid_age_labels = age_labels[valid_mask]
                loss_age = F.cross_entropy(valid_age_logits, valid_age_labels)
            else:
                loss_age = torch.tensor(0.0, device=age_logits.device)

            # 总损失
            total_loss = loss_arcface + self.age_loss_weight * loss_age

            return total_loss, loss_arcface, loss_age
        else:
            # 推理模式：返回归一化的特征向量用于识别
            return F.normalize(features, p=2, dim=1)

In [9]:
def test(num_epochs = 5):
    # 超参数
    embedding_size = 512
    num_classes = len(train_dataset.id_to_label)   # 训练集中的身份数
    num_age_groups = 5   # 年龄组数
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 创建模型
    model = CrossAgeFaceModel(embedding_size, num_classes, num_age_groups).to(device)

    # 优化器：通常对 backbone 和分类头使用不同的学习率
    optimizer = optim.SGD([
        {'params': model.backbone.parameters(), 'lr': 0.01},
        {'params': model.arcface_head.parameters(), 'lr': 0.01},
        {'params': model.age_predictor.parameters(), 'lr': 0.01}
    ], momentum=0.9, weight_decay=5e-4)

    # 学习率调度器
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

    # 训练循环
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (images, labels, age_labels) in enumerate(train_loader):
            images = images.to(device)
            labels = labels.to(device)
            age_labels = age_labels.to(device)

            optimizer.zero_grad()
            total_loss, loss_arc, loss_age = model(images, labels, age_labels)
            total_loss.backward()
            optimizer.step()

            if batch_idx % 1 == 0:
                print(f'Epoch {epoch} Batch {batch_idx}: total={total_loss.item():.4f}, arc={loss_arc.item():.4f}, age={loss_age.item():.4f}')

        scheduler.step()

In [10]:
test(5)

d:\Anaconda3\envs\torch\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\Anaconda3\envs\torch\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 0 Batch 0: total=40.9617, arc=40.8033, age=1.5832
Epoch 0 Batch 1: total=39.5520, arc=39.3920, age=1.5997
Epoch 0 Batch 2: total=39.9079, arc=39.7493, age=1.5865
Epoch 0 Batch 3: total=39.4104, arc=39.2461, age=1.6435
Epoch 0 Batch 4: total=38.5523, arc=38.3912, age=1.6111
Epoch 0 Batch 5: total=38.6366, arc=38.4811, age=1.5546
Epoch 0 Batch 6: total=38.3156, arc=38.1593, age=1.5627
Epoch 0 Batch 7: total=37.3750, arc=37.2274, age=1.4762
Epoch 0 Batch 8: total=38.4282, arc=38.2848, age=1.4341
Epoch 0 Batch 9: total=38.5040, arc=38.3721, age=1.3192
Epoch 0 Batch 10: total=39.2818, arc=39.1442, age=1.3760
Epoch 0 Batch 11: total=38.2599, arc=38.1317, age=1.2819
Epoch 0 Batch 12: total=38.6356, arc=38.5087, age=1.2698
Epoch 0 Batch 13: total=38.7668, arc=38.6420, age=1.2483
Epoch 0 Batch 14: total=37.5880, arc=37.4543, age=1.3367
Epoch 0 Batch 15: total=37.7220, arc=37.5939, age=1.2814
Epoch 0 Batch 16: total=38.3738, arc=38.2603, age=1.1350
Epoch 0 Batch 17: total=38.5268, arc=38.3

KeyboardInterrupt: 